In [ ]:
# ================================
# Basic imports
# ================================
import sys
import warnings
import numpy as np
import pandas as pd
from numpy import loadtxt, savetxt
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d
import matplotlib as mpl
import matplotlib.pyplot as plt

# ================================
# GaPP imports
# ================================
from gapp import gp
from gapp import covariance

# ================================
# MCMC imports
# ================================
import emcee
from multiprocessing import pool
import multiprocessing as mp
from getdist import MCSamples, plots

# ================================
# Matplotlib global configuration
# ================================
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["Nimbus Roman"]
})

# ===================
# Warning control
# ===================
# We suppress only generic UserWarnings to avoid hiding serious numerical issues.
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# =========================
# Loading CC data
# =========================

# Path
cc_dat_path = "/home/brunowesley/projetos/GaPP/Hz_CC_data.txt"

# DataFrame
df_cc = pd.read_csv(cc_dat_path, sep=r"\s+")

# Main columns
Z     = df_cc["z_cc"].to_numpy(float)
Hz    = df_cc["H_cc"].to_numpy(float)
Sigma = df_cc["H_cc_err"].to_numpy(float)


# =========================
# Loading BAO data (DESI_DR2)
# =========================

# Path  
bao_dat_path = "/home/brunowesley/projetos/GaPP/desi_gaussian_bao_ALL_GCcomb_mean.txt"
bao_cov_path = "/home/brunowesley/projetos/GaPP/desi_gaussian_bao_ALL_GCcomb_cov.txt"

# DataFrame
df_bao = pd.read_csv(bao_dat_path, sep=r"\s+", comment="#", names=["z", "value", "quantity"])

# Main columns
z_bao = df_bao["z"].values
d_obs = df_bao["value"].values

# Covariance matrix and its inverse
cov_bao = np.loadtxt(bao_cov_path)
Cinv_bao = np.linalg.inv(cov_bao)

In [ ]:
# Physical constants
c = 299792.458

In [ ]:
# =============================================================================
# 1st part of the work

# Reconstructing H(z) function using CC data
# =============================================================================

In [ ]:
# =============================================================================
# Main execution
# =============================================================================

if __name__ == "__main__":

    # -----------------------------------------
    # Gaussian Process reconstruction of H(z)
    # -----------------------------------------

    # Reconstruction range:
    # from z = 0 up to the maximum redshift in the data
    zmin = 0.0
    zmax = np.max(Z)

    # Instantiate the Gaussian Process with a Squared Exponential kernel
    g1 = gp.GaussianProcess(
        Z,
        Hz,
        Sigma,
        covfunction=covariance.SquaredExponential,
        cXstar=(zmin, zmax, 200)  # 200 reconstruction points between zmin and zmax
    )

    # Perform GP reconstruction and hyperparameter training
    rec1, theta1 = g1.gp(thetatrain="True")


    # -----------------------------------------
    # Extract reconstructed quantities
    # -----------------------------------------
   
    # rec1 columns follow GaPP convention:

    zrec = rec1[:, 0]         # column 0 -> reconstructed z
    hzrec = rec1[:, 1]        # column 1 -> reconstructed H(z)
    sighzrec = rec1[:, 2]     # column 2 -> 1-sigma uncertainty on H(z)


    # -----------------------------------------
    # Print H0 estimate (H at z = 0)
    # -----------------------------------------
    H0 = hzrec[0]
    sigH0 = sighzrec[0]

    print(
        f"z = {zrec[0]:.3f} | "
        f"H0 = {H0:.3f} | "
        f"sigH0 = {sigH0:.3f} | "
        f"sigH0/H0 (%) = {(sigH0 / H0) * 100:.2f}"
    )


    # -----------------------------------------
    # Save reconstructed H(z)
    # -----------------------------------------
    savetxt("Hz_CC_rec.dat", rec1)


    # =========================================================================
    # Plotting: data + GP reconstruction
    # =========================================================================
        
    plt.rc("text", usetex=False)
    plt.rc("font", family="serif")

    fig, ax = plt.subplots(figsize=(9, 7))

    # Axis labels
    ax.set_xlabel(r"$z$", fontsize=22)
    ax.set_ylabel(
        r"$H(z)\;(\mathrm{km}\,\mathrm{s}^{-1}\,\mathrm{Mpc}^{-1})$",
        fontsize=22
    )

    # Axis limits
    ax.set_xlim(zmin, zmax + 0.1)

    # Tick font sizes
    ax.tick_params(axis="both", labelsize=22)

    # Plot observational data
    data_plot = ax.errorbar(
        Z, Hz, yerr=Sigma,
        fmt="o",
        color="black",
        label=r"$H(z)$ data"
    )

    # Plot GP confidence regions
    band1 = ax.fill_between(
        zrec,
        hzrec - sighzrec,
        hzrec + sighzrec,
        facecolor="#F08080",
        alpha=0.80,
        label=r"$1\sigma$"
    )

    band2 = ax.fill_between(
        zrec,
        hzrec - 2.0 * sighzrec,
        hzrec + 2.0 * sighzrec,
        facecolor="#F08080",
        alpha=0.50,
        label=r"$2\sigma$"
    )

    band3 = ax.fill_between(
        zrec,
        hzrec - 3.0 * sighzrec,
        hzrec + 3.0 * sighzrec,
        facecolor="#F08080",
        alpha=0.30,
        label=r"$3\sigma$"
    )

    # Legend (explicit order for robustness)
    ax.legend(
        [band1, band2, band3, data_plot],
        [r"$1\sigma$", r"$2\sigma$", r"$3\sigma$", r"$H(z)$ data"],
        fontsize=22,
        loc="upper left"
    )

    # Save figure before showing it
    fig.savefig(
        "Hz_CC_rec.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# =============================================================================
# 2nd part of the work

# Reconstructing DM(z), DH(z), and DV(z) BAO 
# quantities from the H(z) reconstructed above
# =============================================================================

In [ ]:
# =========================================================================
# Cosmological distances from reconstructed H(z)
# =========================================================================


# --------------------------------------------------------------
# Hubble distance
# --------------------------------------------------------------
# DH(z) = c / H(z)
DH = c / hzrec

# sigma_DH / DH = sigma_H / H
sigDH = DH * (sighzrec / hzrec) 


# --------------------------------------------------------------
# Comoving transverse distance
# --------------------------------------------------------------
# DM(z) = c * ∫ dz / H(z)
integrand = 1.0 / hzrec
DM = c * cumulative_trapezoid(integrand, zrec, initial=0.0)

# σ_DM^2 = c^2 ∫ [σ_H(z) / H(z)^2]^2 dz
integrand_err = (sighzrec / hzrec**2) ** 2
sigDM = c * np.sqrt(cumulative_trapezoid(integrand_err, zrec, initial=0.0))


# --------------------------------------------------------------
# Volume-averaged distance
# --------------------------------------------------------------
# DV(z) = [ z * DM^2 * DH ]^(1/3)
DV = (zrec * DM**2 * DH) ** (1.0 / 3.0)

# Error propagation (logarithmic differentiation)
sigDV = DV * np.sqrt(
    (2.0 / 3.0 * sigDM / DM) ** 2
    + (1.0 / 3.0 * sigDH / DH) ** 2
)


# Avoid numerical issues at z = 0
DV[0] = 0.0
sigDV[0] = 0.0

In [ ]:
# =========================================================================
# Save reconstructed distances
# =========================================================================

savetxt(
    "BAO_dist_rec.dat",
    np.column_stack([zrec, DH, sigDH, DM, sigDM, DV, sigDV]),
    header="z  DH  sigDH  DM  sigDM  DV  sigDV"
)

In [ ]:
# =========================================================================
# Combined plot: DH(z), DM(z), DV(z) in custom layout (A, B, C)
# =========================================================================

fig = plt.figure(figsize=(12, 10))

# Create GridSpec: 2 rows, 2 columns
gs = fig.add_gridspec(
    nrows=2,
    ncols=2,
    height_ratios=[1, 1.2],  # bottom plot slightly taller
    hspace=0.25,
    wspace=0.35
)

# Axes definition
ax_DH = fig.add_subplot(gs[0, 0])   # Plot A
ax_DM = fig.add_subplot(gs[0, 1])   # Plot B
ax_DV = fig.add_subplot(gs[1, 0])   # Plot C (single column, will be centered)

# -------------------------------------------------------------------------
# Center Plot C horizontally
# -------------------------------------------------------------------------
pos = ax_DV.get_position()
new_width = pos.width
new_x0 = 0.5 - new_width / 2.0   # center horizontally

ax_DV.set_position([new_x0, pos.y0, new_width, pos.height])


# =========================================================================
# Plot A: DH(z)
# =========================================================================
ax_DH.plot(zrec, DH, color="black", ls="--", lw=1.2)

ax_DH.fill_between(zrec, DH - 3*sigDH, DH + 3*sigDH,
                   color="#D62728", alpha=0.25)
ax_DH.fill_between(zrec, DH - 2*sigDH, DH + 2*sigDH,
                   color="#D62728", alpha=0.45)
ax_DH.fill_between(zrec, DH - sigDH, DH + sigDH,
                   color="#D62728", alpha=0.65)

ax_DH.set_ylabel(r"$D_H(z)\;(\mathrm{Mpc})$", fontsize=16)
ax_DH.tick_params(axis="both", labelsize=12)

# =========================================================================
# Plot B: DM(z)
# =========================================================================
ax_DM.plot(zrec, DM, color="black", ls="--", lw=1.2)

ax_DM.fill_between(zrec, DM - 3*sigDM, DM + 3*sigDM,
                   color="#1f77b4", alpha=0.25)
ax_DM.fill_between(zrec, DM - 2*sigDM, DM + 2*sigDM,
                   color="#1f77b4", alpha=0.45)
ax_DM.fill_between(zrec, DM - sigDM, DM + sigDM,
                   color="#1f77b4", alpha=0.65)

ax_DM.set_ylabel(r"$D_M(z)\;(\mathrm{Mpc})$", fontsize=16)
ax_DM.tick_params(axis="both", labelsize=12)

# =========================================================================
# Plot C: DV(z)
# =========================================================================
ax_DV.plot(zrec, DV, color="black", ls="--", lw=1.2)

ax_DV.fill_between(zrec, DV - 3*sigDV, DV + 3*sigDV,
                   color="#2CA02C", alpha=0.25)
ax_DV.fill_between(zrec, DV - 2*sigDV, DV + 2*sigDV,
                   color="#2CA02C", alpha=0.45)
ax_DV.fill_between(zrec, DV - sigDV, DV + sigDV,
                   color="#2CA02C", alpha=0.65)

ax_DV.set_xlabel(r"$z$", fontsize=16)
ax_DV.set_ylabel(r"$D_V(z)\;(\mathrm{Mpc})$", fontsize=16)
ax_DV.tick_params(axis="both", labelsize=12)

# =========================================================================
# Axis limits
# =========================================================================
ax_DH.set_xlim(zrec[0], zrec[-1])
ax_DM.set_xlim(zrec[0], zrec[-1])
ax_DV.set_xlim(zrec[0], zrec[-1])

# =========================================================================
# Save and show
# =========================================================================
fig.savefig(
    "BAO_dist_rec.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================
# 3rd part of the work

# BAO analysis using distances reconstructed
# from the GP H(z)
# =============================================================================

In [ ]:
# =====================================================
# Interpolation of reconstructed distance functions
# =====================================================

# The GP reconstruction provides DH(z), DM(z) and DV(z) only at the
# discrete redshift grid zrec. In order to evaluate these distances
# at the specific redshifts of the BAO measurements, we construct
# smooth interpolating functions.

# Cubic interpolation is adopted to ensure continuity of the first
# and second derivatives, which is desirable for cosmological
# distance functions.

# Note: the interpolation is strictly valid only within the redshift
# range covered by the CC data used in the GP reconstruction.

DM_gp = interp1d(zrec, DM, kind="cubic")
DH_gp = interp1d(zrec, DH, kind="cubic")
DV_gp = interp1d(zrec, DV, kind="cubic")

In [ ]:
# =====================================================
# Redshift consistency between CC reconstruction and BAO data
# =====================================================

# The H(z) reconstruction based on cosmic chronometers is limited
# to z <= zmax_CC = max(zrec). BAO measurements at higher redshifts
# would require extrapolation of the GP reconstruction, which is
# not statistically controlled and may introduce biases.

# To ensure a fully data-driven and conservative analysis, we
# restrict the BAO dataset to the redshift range covered by the
# reconstructed distances.

# Mask z <= zrec max.
mask = df_bao["z"] <= zrec.max()
df_bao = df_bao[mask].reset_index(drop=True)

# Updated BAO data
z_bao = df_bao["z"].values
d_obs = d_obs[mask]

# Updated BAO covariance
cov_bao = cov_bao[np.ix_(mask, mask)]
Cinv_bao = np.linalg.inv(cov_bao)

In [ ]:
# =======================================
# Sound horizon fitting formula
# =======================================

# omega_i = Omega_i * h**2 at z=0,
# where h = H0 / 100 (dimensionless hubble parameter) 

def rd_fit(omega_m, omega_b, Neff):
    
    rd = (
        147.05
        * (omega_b / 0.02236)**-0.13
        * (omega_m / 0.1432)**-0.23
        * (Neff / 3.04)**-0.1
        )

    return rd

In [ ]:
# ==============================
# BAO functions
# ==============================

# Transverse BAO distance scaled by rd
def DM_over_rd(z, omega_m, omega_b, Neff):
    rd_ff = rd_fit(omega_m, omega_b, Neff)
    return DM_gp(z) / rd_ff


# Hubble distance scaled by rd
def DH_over_rd(z, omega_m, omega_b, Neff):
    rd_ff = rd_fit(omega_m, omega_b, Neff)
    return DH_gp(z) / rd_ff


# Volume-averaged distance scaled by rd
def DV_over_rd(z, omega_m, omega_b, Neff):

    dm = DM_over_rd(z, omega_m, omega_b, Neff)
    dh = DH_over_rd(z, omega_m, omega_b, Neff)

    return (z * dm**2 * dh)**(1.0 / 3.0)


# ==============================
# Theoretical BAO distance vector (following the d_obs sequence)
# ==============================

def d_GP(z, omega_m, omega_b, Neff):

    # List to store the model predictions in the same order as the data
    d_gp = []

    # Loop over BAO measurements
    for _, row in df_bao.iterrows():
        z = row["z"]                 # Redshift of the BAO measurement
        q = row["quantity"]          # Type of BAO observable

        # Select the appropriate theoretical BAO distance
        if q == "DV_over_rs":
            d = DV_over_rd(z, omega_m, omega_b, Neff)

        elif q == "DM_over_rs":
            d = DM_over_rd(z, omega_m, omega_b, Neff)

        elif q == "DH_over_rs":
            d = DH_over_rd(z, omega_m, omega_b, Neff)

        else:
            raise ValueError(f"Observável BAO desconhecido: {q}")

        d_gp.append(d)

    # Return model BAO vector as a NumPy array
    return np.array(d_gp)


# ========================
# Sanity test
# ========================

print("d_GP(z_bao) =", d_GP(z_bao, 0.1432, 0.02236, 3.04))

In [ ]:
# =========================================
# BAO Likelihood (DESI_DR2)
# =========================================

# Param priors
om_min, om_max = 0.1419, 0.1445
ob_min, ob_max = 0.017784, 0.031616
Neff_min, Neff_max = 2.82, 3.26

# Log-priors
def lnprior_bao(theta_bao):

    # Params vector
    omega_m, omega_b, Neff = theta_bao
    
    # Flat priors
    if not (om_min < omega_m < om_max):     return -np.inf
    if not (ob_min < omega_b < ob_max):     return -np.inf
    if not (Neff_min < Neff < Neff_max):    return -np.inf

    # Gaussian prior
    # lp_Ob = -0.5 * ((omega_b - mu_ob)**2 / sigma_ob**2) - np.log(sigma_ob * np.sqrt(2*np.pi))
    
    return 0.0 # lp_Neff


# Log-likelihood
def lnlike_bao(theta_bao, z_bao, d_obs, Cinv_bao):
    omega_m, omega_b, Neff = theta_bao

    # Theoretical BAO distance vector
    dgp_model = d_GP(z_bao, omega_m, omega_b, Neff)

    # Residual vector
    delta = d_obs - dgp_model

    # Chi-squared
    chi2_bao = np.dot(delta, np.dot(Cinv_bao, delta))

    return -0.5 * chi2_bao


# Log-posterior
def lnprob_bao(theta_bao, z_bao, d_obs, Cinv_bao):

    # Log-prior
    lp = lnprior_bao(theta_bao)
    if not np.isfinite(lp):
        return -np.inf
    
    # Log-likelihood
    ll = lnlike_bao(theta_bao, z_bao, d_obs, Cinv_bao)

    return lp + ll


# =========================
# Quick test
# =========================

theta_bao_test = [0.1432, 0.02236, 3.04]
print("GP_BAO log-posterior =", lnprob_bao(theta_bao_test, z_bao, d_obs, Cinv_bao))

In [ ]:
# =========================
# MCMC configuration
# =========================

# Dimensions and sampling settings
ndim, nwalkers, nsteps, nburn = 3, 40, 53500, 3500
rng = np.random.default_rng(42)

# Initial walker positions drawn from priors
p0 = np.zeros((nwalkers, ndim))
p0[:,0] = rng.uniform(om_min, om_max, size=nwalkers)         # omega_m uniforme
p0[:,1] = rng.uniform(ob_min, ob_max, size=nwalkers)         # Omega_b uniforme
p0[:,2] = rng.uniform(Neff_min, Neff_max, size=nwalkers)     # Neff uniforme
# p0[:,2] = rng.normal(mu_Neff, sigma_Neff, nwalkers)          # Neff gaussiano


# Parallel chain generation (MORE efficient for GP-BAO likelihood)
with mp.Pool(processes=nwalkers) as pool:
    sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob_bao, args=(z_bao, d_obs, Cinv_bao), pool=pool)
    sampler.run_mcmc(p0, nsteps, progress=True)

# Single-threaded chain generation (LESS efficient for GP-BAO likelihood)
# sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob_bao, args=(z_bao, d_obs, Cinv_bao))
# sampler.run_mcmc(p0, nsteps, progress=True)


# Full chain: shape (nsteps, nwalkers, ndim)
chain = sampler.get_chain()
# np.save("chain_LCDM_bao.npy", chain)

# Flattened chain after burn-in: shape (N_total, ndim)
flat_samples = sampler.get_chain(discard=nburn, flat=True)
np.save("flat_samples_gp_bao.npy", flat_samples)

In [ ]:
# =========================
# Corner plot using GetDist
# =========================

# Parameter names and LaTeX labels
param_names  = ["Omega_m", "Omega_b", "Neff"]
param_labels = [r"\Omega_m h^2", r"\Omega_b h^2", r"N_{eff}"]


# Create GetDist samples object
samples = MCSamples(
    samples=flat_samples,
    names=param_names,
    labels=param_labels
)

# Smoothing and binning settings
samples.updateSettings({
    "smooth_scale_1D": 0.25,
    "smooth_scale_2D": 0.25,
    "fine_bins": 1024,
    "fine_bins_2D": 1024

})

# Parameter ranges (optional)
# samples.setRanges({
#     "param p1": (p1_min, p1_max),
#     "param p2": (p2_min, p2_max),
#     "param p3": (p3_min, p3_max)
# })


# Initialize GetDist subplot plotter
g = plots.get_subplot_plotter()

# Plot styling settings
g.settings.axes_fontsize = 12
g.settings.lab_fontsize = 14
g.settings.legend_fontsize = 10
g.settings.linewidth_contour = 1.2
g.settings.num_plot_contours = 2
g.settings.axis_marker_lw = 1.0
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add = 0.3


# Triangle (corner) plot
g.triangle_plot(
    samples,
    filled=True,
    legend_labels=["GP + DESI_DR2 Data"],
    #contour_colors=["#"],
    title_limit=1
)

# Save figure
plt.savefig("Params_fit_gp_bao.png", dpi=300, bbox_inches="tight")
plt.show()